# Probabilistic Calibration via Conformalized Quantile Regression (Week 5, Task 3)

## PRE-REGISTRATION (declared before results)

### Goal
Build a probabilistic 30-day-ahead forecast for corn returns using quantile regression 
(q10, q50, q90) + conformal calibration to achieve 80% empirical coverage. 
Evaluate coverage OVERALL, per REGIME, and per YEAR. Use vanilla features only 
(no regime drivers, as regime features did not lift point accuracy in Task 2).

### Feature Set (FROZEN — VANILLA ONLY)
- ret_1, ret_5, ret_10, ret_21, ret_63 (lagged log returns)
- vol_21 (21-day rolling realized volatility)
- sin_doy, cos_doy (seasonal day-of-year indicators)
Total: 8 features

### Quantile Regression Setup
- **Model**: XGBoost with `reg:quantileerror` loss
- **Quantiles**: q10, q50 (median), q90
- **Implementation**: 
  - If XGBoost >= 2.0 supports multi-quantile in one call, use it
  - Otherwise fit three separate models (one per quantile)
- **Constraint**: Enforce monotonic sort post-prediction: q10 ≤ q50 ≤ q90 per row (no quantile crossing)
- **XGBoost hyperparameters** (IDENTICAL to Week-4 vanilla baseline):
  - n_estimators=300, max_depth=3, learning_rate=0.03
  - subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, random_state=0

### Conformal Quantile Regression (CQR) Setup
- **Calibration target**: 80% empirical coverage (alpha = 0.2)
- **Calibration data**: Recent trailing ~1 year (~252 trading days) of in-fold training data
- **Tool**: mapie's CQR class (version-aware)
- **Decision gate** (pre-registered): 
  1. **Marginal CQR first**: Single calibration, single correction Q
  2. Measure per-regime coverage
  3. **Promote-to-conditional rule**:
     - If all regimes within ~70-90% coverage → **MARGINAL STANDS** (ship it)
     - If any regime under-covers (< 70%, esp. high-vol regimes) → **PROMOTE to VOL-TERCILE CONDITIONAL**:
       Bucket by vol_21 into 3 terciles, apply separate CQR per tercile
  4. Document choice + numbers

### Walk-Forward Evaluation
- **Folds**: Reuse Task-2 harness (expanding window, step=21, embargo=30, OOS 2020-2026)
- **Per fold**:
  1. Split in-fold data: older chunk → model-train, recent ~252 days → conformal-calibration
  2. Fit q10/q50/q90 quantile models on train
  3. Conformalize on calib
  4. Predict intervals on test
- **Output**: OOS results with [low, high, realized, covered, width] per point

### Coverage Measurement (THE DELIVERABLE)
For every OOS test point, record:
- Predicted interval [q10, q90] (after CQR adjustment)
- Realized outcome
- Covered = (outcome ∈ [q10, q90])
- Width = q90 - q10

Then compute + report:
- **Empirical coverage OVERALL** (% of OOS points covered)
- **Per-regime coverage** (group OOS by the seven PELT regime labels)
- **Per-year coverage** (group by calendar year)
- **Mean width OVERALL and per-regime** (coverage + width reported together)

Target: 80% coverage. Width should be reasonable (not bloated by over-correction).

### Final Production Model
Fit quantile models + CQR on ALL data through latest date, serialize to joblib:
- `quantile_models.joblib`: (q10_model, q50_model, q90_model)
- `conformal_calibration.joblib`: CQR calibration object
These are committed to the repo for Render to load at startup.

### Transform Note
The model predicts 30-day log RETURN quantiles. The live API contract expects PRICE quantiles.
Transformation (documented, not implemented here):
  `price_low = last_price * exp(q10)`
  `price_point = last_price * exp(q50)`
  `price_high = last_price * exp(q90)`

In [14]:
import importlib.util
import numpy as np
import pandas as pd
import duckdb
import xgboost as xgb
from pathlib import Path

# mapie import (version-aware)
mapie_v1_spec = importlib.util.find_spec("mapie.regression")
if mapie_v1_spec is not None:
    MAPIE_V1 = True
    print("✓ mapie v1.x detected")
else:
    MAPIE_V1 = False
    print("✓ mapie v0.x detected")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print(f"Root: {ROOT}")

✓ mapie v1.x detected
Root: c:\dev\openag-risk-twin


In [15]:
# Load panel from DuckDB
con = duckdb.connect(str(ROOT / "data" / "openag.duckdb"), read_only=True)
panel = con.sql('SELECT * FROM panel ORDER BY "Date"').df()
con.close()

panel["Date"] = pd.to_datetime(panel["Date"]).astype("datetime64[ns]")
panel = panel.set_index("Date")

print(f"Panel: {panel.shape[0]} rows, {panel.index.min().date()} → {panel.index.max().date()}")
print(f"Columns: {list(panel.columns)}")
print("\nRegime distribution:")
print(panel["regime"].value_counts().sort_index())

Panel: 2512 rows, 2016-06-06 → 2026-06-05
Columns: ['Close', 'log_ret', 'rolling_vol', 'spec_net', 'spec_net_pct_oi', 'drought_stress', 'usd_broad', 'treasury_10y', 'wti_crude', 'corn_ending_stocks_mmt', 'regime']

Regime distribution:
regime
breakout_21        70
burst_spr21        65
burst_spr23        55
calm_16_21       1150
cooling_22_23     160
high_21_22        295
low_23_26         717
Name: count, dtype: int64


In [16]:
H = 30  # forecast horizon

df = panel.copy()

# Target: 30-day-ahead log return
df["y"] = np.log(df["Close"].shift(-H)) - np.log(df["Close"])

# Vanilla features (unchanged from Week 4)
for lag in [1, 5, 10, 21, 63]:
    df[f"ret_{lag}"] = np.log(df["Close"]) - np.log(df["Close"].shift(lag))

df["vol_21"] = df["rolling_vol"]

doy = df.index.dayofyear
df["sin_doy"] = np.sin(2 * np.pi * doy / 365.25)
df["cos_doy"] = np.cos(2 * np.pi * doy / 365.25)

FEATURES_VANILLA = [
    "ret_1", "ret_5", "ret_10", "ret_21", "ret_63",
    "vol_21", "sin_doy", "cos_doy"
]

print(f"Vanilla features: {len(FEATURES_VANILLA)}")
print(f"Target (y) non-NaN: {df['y'].notna().sum()}")

# Coverage: rows with no NaN in vanilla features or target
model_df = df.dropna(subset=FEATURES_VANILLA + ["y"])
print(f"Model-ready rows: {len(model_df)}")


Vanilla features: 8
Target (y) non-NaN: 2482
Model-ready rows: 2419


In [17]:
def walk_forward_folds(idx, oos_start, step=21, embargo=H, min_train=250):
    """Expanding-window folds; train target windows never reach test region (embargo=H)."""
    idx = pd.DatetimeIndex(idx)
    start = int(np.searchsorted(idx, pd.Timestamp(oos_start)))
    folds = []
    for origin in range(start, len(idx), step):
        train_pos = np.arange(0, origin - embargo)
        test_pos = np.arange(origin, min(origin + step, len(idx)))
        if len(train_pos) >= min_train and len(test_pos) > 0:
            folds.append((train_pos, test_pos))
    return folds


def evaluate(folds, data, features, fit_predict_fn):
    """Run fit_predict_fn(train_df, test_df, features) -> yhat across folds; stack OOS predictions."""
    parts = []
    for train_pos, test_pos in folds:
        tr, te = data.iloc[train_pos], data.iloc[test_pos]
        result = fit_predict_fn(tr, te, features)
        # result can be a DataFrame (with q10/q50/q90 columns) or array (point forecast)
        if isinstance(result, pd.DataFrame):
            pred_df = result.copy()
        else:
            pred_df = pd.DataFrame({"yhat": np.asarray(result)}, index=te.index)
        
        # Add true targets
        pred_df["y"] = te["y"].to_numpy()
        parts.append(pred_df)
    
    return pd.concat(parts)


# Create folds (identical to Task 2)
folds = walk_forward_folds(model_df.index, oos_start="2020-01-01")
print(f"Folds: {len(folds)} expanding windows")
print(f"OOS period: {model_df.index[folds[0][1][0]].date()} → {model_df.index[-1].date()}")


Folds: 76 expanding windows
OOS period: 2020-01-02 → 2026-04-23


In [18]:
def enforce_quantile_monotonicity(q_low, q_mid, q_high):
    """Ensure q_low <= q_mid <= q_high via sorting per row."""
    quantiles = np.column_stack([q_low, q_mid, q_high])
    sorted_q = np.sort(quantiles, axis=1)
    return sorted_q[:, 0], sorted_q[:, 1], sorted_q[:, 2]


# Test: verify the function works
test_q10 = np.array([0.01, -0.02, 0.00])
test_q50 = np.array([-0.01, 0.00, 0.01])
test_q90 = np.array([0.05, 0.02, 0.03])
q10_s, q50_s, q90_s = enforce_quantile_monotonicity(test_q10, test_q50, test_q90)
print("Monotonic sort test:")
print(f"  Input:  q10={test_q10}, q50={test_q50}, q90={test_q90}")
print(f"  Output: q10={q10_s}, q50={q50_s}, q90={q90_s}")
assert np.all(q10_s <= q50_s) and np.all(q50_s <= q90_s), "Monotonicity enforced ✓"

Monotonic sort test:
  Input:  q10=[ 0.01 -0.02  0.  ], q50=[-0.01  0.    0.01], q90=[0.05 0.02 0.03]
  Output: q10=[-0.01 -0.02  0.  ], q50=[0.01 0.   0.01], q90=[0.05 0.02 0.03]


In [19]:

print(f"XGBoost version: {xgb.__version__}")

# Check if multi-quantile in one model is supported (XGBoost >= 2.0)
# For now, use three separate models for robustness across versions
MULTI_QUANTILE_SUPPORTED = tuple(map(int, xgb.__version__.split('.')[:2])) >= (2, 0)
print(f"Multi-quantile in one call: {MULTI_QUANTILE_SUPPORTED}")

XGB_PARAMS = dict(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    base_score=0.0, 
    random_state=0,
    n_jobs=-1,
)
print(f"XGB params (shared across quantiles): {XGB_PARAMS}")


XGBoost version: 3.2.0
Multi-quantile in one call: True
XGB params (shared across quantiles): {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'base_score': 0.0, 'random_state': 0, 'n_jobs': -1}


In [20]:
def fit_predict_quantiles(tr, te, features, quantiles=[0.1, 0.5, 0.9]):
    """Fit separate XGBoost quantile regressors per quantile; predict on test."""
    q_preds = {}
    
    for q in quantiles:
        model = xgb.XGBRegressor(
            objective="reg:quantileerror",
            quantile_alpha=q,
            **XGB_PARAMS
        )
        model.fit(tr[features], tr["y"])
        q_preds[q] = model.predict(te[features])
    
    q10 = q_preds[0.1]
    q50 = q_preds[0.5]
    q90 = q_preds[0.9]
    
    # Enforce monotonic sort
    q10, q50, q90 = enforce_quantile_monotonicity(q10, q50, q90)
    
    return pd.DataFrame(
        {
            "q10": q10,
            "q50": q50,
            "q90": q90,
        },
        index=te.index,
    )


# Test on a small fold
test_fold_idx = 0
tr_pos, te_pos = folds[test_fold_idx]
tr, te = model_df.iloc[tr_pos], model_df.iloc[te_pos]

print(f"Test fold {test_fold_idx}:")
print(f"  Train: {len(tr)} rows, {tr.index.min().date()} → {tr.index.max().date()}")
print(f"  Test:  {len(te)} rows, {te.index.min().date()} → {te.index.max().date()}")

test_result = fit_predict_quantiles(tr, te, FEATURES_VANILLA)
print("\nQuantile predictions (first 5 rows):")
print(test_result.head())
print("\nMonotonicity check:")
print(f"  q10 <= q50: {(test_result['q10'] <= test_result['q50']).all()}")
print(f"  q50 <= q90: {(test_result['q50'] <= test_result['q90']).all()}")

print("\nEmpirical training quantiles of y (fold 0):")
print(tr["y"].quantile([0.1, 0.5, 0.9]).round(4).to_dict())
print("Model predicted quantiles (fold 0, mean over test rows):")
print(test_result.mean().round(4).to_dict())

Test fold 0:
  Train: 803 rows, 2016-09-02 → 2019-11-15
  Test:  21 rows, 2020-01-02 → 2020-01-31

Quantile predictions (first 5 rows):
                 q10       q50       q90
Date                                    
2020-01-02 -0.022671  0.034976  0.053481
2020-01-03 -0.009172  0.048163  0.062486
2020-01-06 -0.011244  0.047333  0.057132
2020-01-07 -0.024497  0.042955  0.057060
2020-01-08 -0.024142  0.043968  0.057899

Monotonicity check:
  q10 <= q50: True
  q50 <= q90: True

Empirical training quantiles of y (fold 0):
{0.1: -0.0799, 0.5: 0.0115, 0.9: 0.0701}
Model predicted quantiles (fold 0, mean over test rows):
{'q10': -0.028699999675154686, 'q50': 0.0333000011742115, 'q90': 0.057500001043081284}


In [21]:
def fit_predict_with_cqr(tr, te, features, alpha=0.2, calib_size=252):
    """
    Split tr into model-train (older) + calib (recent ~1yr), fit quantiles,
    conformalize on calib, predict intervals on te. Point anchored at RW (0).
    """
    calib_start_idx = max(0, len(tr) - calib_size)
    tr_model = tr.iloc[:calib_start_idx]
    tr_calib = tr.iloc[calib_start_idx:]

    # Fit quantile models on tr_model
    models = {}
    for q in [0.1, 0.5, 0.9]:
        model = xgb.XGBRegressor(
            objective="reg:quantileerror",
            quantile_alpha=q,
            **XGB_PARAMS
        )
        model.fit(tr_model[features], tr_model["y"])
        models[q] = model

    # Predict on calibration set
    q10_calib = models[0.1].predict(tr_calib[features])
    q50_calib = models[0.5].predict(tr_calib[features])
    q90_calib = models[0.9].predict(tr_calib[features])
    q10_calib, q50_calib, q90_calib = enforce_quantile_monotonicity(q10_calib, q50_calib, q90_calib)

    # Predict on test set  (this block was missing)
    q10_test = models[0.1].predict(te[features])
    q50_test = models[0.5].predict(te[features])
    q90_test = models[0.9].predict(te[features])
    q10_test, q50_test, q90_test = enforce_quantile_monotonicity(q10_test, q50_test, q90_test)

    y_calib = tr_calib["y"].values

    # Recenter both bands on their own median -> point = RW (0)
    lower_calib = q10_calib - q50_calib
    upper_calib = q90_calib - q50_calib
    q10_test = q10_test - q50_test
    q90_test = q90_test - q50_test
    q50_test = np.zeros_like(q50_test)

    # CQR: one signed score per calib point, one quantile -> one Q
    scores = np.maximum(lower_calib - y_calib, y_calib - upper_calib)
    n = len(scores)
    k = min(int(np.ceil((1 - alpha) * (n + 1))), n)
    Q = np.sort(scores)[k - 1]

    q10_corrected = q10_test - Q
    q90_corrected = q90_test + Q

    return pd.DataFrame(
        {"q10": q10_corrected, "q50": q50_test, "q90": q90_corrected},
        index=te.index,
    )


# Run walk-forward evaluation with CQR
print("Running walk-forward evaluation with marginal CQR...")
print("(This may take 10-15 minutes for 76 folds)")

oos_results = []
for i, (tr_pos, te_pos) in enumerate(folds):
    if i % 10 == 0:
        print(f"  Fold {i}/{len(folds)}...")
    
    tr, te = model_df.iloc[tr_pos], model_df.iloc[te_pos]
    pred_df = fit_predict_with_cqr(tr, te, FEATURES_VANILLA)
    pred_df["y"] = te["y"].values
    pred_df["regime"] = te["regime"].values
    
    oos_results.append(pred_df)

oos = pd.concat(oos_results)
print(f"\nOOS results: {len(oos)} points")
print(oos.head(10))

Running walk-forward evaluation with marginal CQR...
(This may take 10-15 minutes for 76 folds)
  Fold 0/76...
  Fold 10/76...
  Fold 20/76...
  Fold 30/76...
  Fold 40/76...
  Fold 50/76...
  Fold 60/76...
  Fold 70/76...

OOS results: 1586 points
                 q10  q50       q90         y      regime
Date                                                     
2020-01-02 -0.116994  0.0  0.089635 -0.035753  calm_16_21
2020-01-03 -0.111918  0.0  0.092775 -0.009097  calm_16_21
2020-01-06 -0.110629  0.0  0.091472 -0.011108  calm_16_21
2020-01-07 -0.117023  0.0  0.094243 -0.015728  calm_16_21
2020-01-08 -0.119329  0.0  0.093217 -0.019048  calm_16_21
2020-01-09 -0.115285  0.0  0.088587 -0.029122  calm_16_21
2020-01-10 -0.120066  0.0  0.094925 -0.034952  calm_16_21
2020-01-13 -0.118074  0.0  0.103144 -0.050010  calm_16_21
2020-01-14 -0.116369  0.0  0.094886 -0.065053  calm_16_21
2020-01-15 -0.116184  0.0  0.095259 -0.055717  calm_16_21


In [22]:
def compute_coverage(oos):
    """Compute empirical coverage: overall, per-regime, per-year."""
    oos["covered"] = (oos["y"] >= oos["q10"]) & (oos["y"] <= oos["q90"])
    oos["width"] = oos["q90"] - oos["q10"]
    
    # Overall
    overall_coverage = oos["covered"].mean()
    overall_width = oos["width"].mean()
    
    # Per regime
    regime_coverage = oos.groupby("regime", observed=True)["covered"].agg(["mean", "count"])
    regime_coverage.columns = ["coverage", "count"]
    regime_width = oos.groupby("regime", observed=True)["width"].mean()
    regime_stats = pd.concat([regime_coverage, regime_width.rename("width")], axis=1)
    
    # Per year
    oos["year"] = oos.index.year
    year_coverage = oos.groupby("year", observed=True)["covered"].agg(["mean", "count"])
    year_coverage.columns = ["coverage", "count"]
    year_width = oos.groupby("year", observed=True)["width"].mean()
    year_stats = pd.concat([year_coverage, year_width.rename("width")], axis=1)
    
    return {
        "overall_coverage": overall_coverage,
        "overall_width": overall_width,
        "regime_stats": regime_stats,
        "year_stats": year_stats,
    }


marginal_coverage = compute_coverage(oos)

print("=" * 80)
print("MARGINAL CQR COVERAGE (OVERALL)")
print("=" * 80)
print(f"Coverage: {marginal_coverage['overall_coverage']:.2%} (target: 80.00%)")
print(f"Width:    {marginal_coverage['overall_width']:.4f}")
print()

print("=" * 80)
print("COVERAGE BY REGIME")
print("=" * 80)
regime_stats = marginal_coverage["regime_stats"].copy()
regime_stats["coverage"] = regime_stats["coverage"] * 100
print(regime_stats.to_string())
print()

print("=" * 80)
print("COVERAGE BY YEAR")
print("=" * 80)
year_stats = marginal_coverage["year_stats"].copy()
year_stats["coverage"] = year_stats["coverage"] * 100
print(year_stats.to_string())

MARGINAL CQR COVERAGE (OVERALL)
Coverage: 80.83% (target: 80.00%)
Width:    0.2478

COVERAGE BY REGIME
                coverage  count     width
regime                                   
breakout_21    74.285714     70  0.296108
burst_spr21    60.000000     65  0.341401
burst_spr23    36.363636     55  0.209060
calm_16_21     67.716535    254  0.251581
cooling_22_23  94.375000    160  0.269291
high_21_22     80.000000    295  0.315306
low_23_26      89.082969    687  0.201645

COVERAGE BY YEAR
       coverage  count     width
year                            
2020  67.588933    253  0.251387
2021  82.539683    252  0.315595
2022  76.494024    251  0.299614
2023  81.200000    250  0.225056
2024  84.523810    252  0.223361
2025  90.438247    251  0.197470
2026  88.311688     77  0.162338


## DECISION GATE: Marginal vs. Conditional CQR

### Rule (declared BEFORE examining results):
1. **Measure per-regime coverage** from marginal CQR above
2. **Check coverage balance**:
   - If **all regimes within 70–90%**: Marginal CQR stands (ship it)
   - If **any regime < 70%** (esp. high-vol regimes like burst_spr21/burst_spr23): Promote to VOL-TERCILE CONDITIONAL
3. **Conditional approach** (if needed):
   - Bucket OOS test by vol_21 into 3 terciles (low, mid, high volatility)
   - Apply separate CQR calibration + correction per tercile
   - Target: achieve ~80% coverage within each tercile

### Rationale
- Thin buckets (7 regimes) lead to unstable calibration + bloated intervals
- Vol terciles (3 buckets) are coarse, stable, and have causal grounding
- High-vol regimes need wider intervals to achieve target coverage

In [23]:
def fit_predict_with_vol_tercile_cqr(tr, te, features, alpha=0.2, calib_size=252):
    """CQR conditioned on vol_21 terciles. Point anchored at RW (0)."""
    calib_start_idx = max(0, len(tr) - calib_size)
    tr_model = tr.iloc[:calib_start_idx]
    tr_calib = tr.iloc[calib_start_idx:]

    models = {}
    for q in [0.1, 0.5, 0.9]:
        m = xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=q, **XGB_PARAMS)
        m.fit(tr_model[features], tr_model["y"])
        models[q] = m

    # Predict on test, enforce monotonicity, recenter on RW (0)
    q10_test = models[0.1].predict(te[features])
    q50_test = models[0.5].predict(te[features])
    q90_test = models[0.9].predict(te[features])
    q10_test, q50_test, q90_test = enforce_quantile_monotonicity(q10_test, q50_test, q90_test)
    q10_test = q10_test - q50_test
    q90_test = q90_test - q50_test

    vol_terciles = np.percentile(tr_calib["vol_21"], [33.33, 66.67])
    te_vol_tercile = np.digitize(te["vol_21"], vol_terciles)
    calib_vol_tercile = np.digitize(tr_calib["vol_21"], vol_terciles)

    q10_corrected = q10_test.copy()
    q90_corrected = q90_test.copy()

    for tercile in range(3):
        mask = te_vol_tercile == tercile
        if not mask.any():
            continue
        calib_mask = calib_vol_tercile == tercile
        if calib_mask.sum() < 20:
            continue

        sub = tr_calib.iloc[calib_mask]
        y_calib_t = sub["y"].values
        q10_c = models[0.1].predict(sub[features])
        q50_c = models[0.5].predict(sub[features])
        q90_c = models[0.9].predict(sub[features])
        q10_c, q50_c, q90_c = enforce_quantile_monotonicity(q10_c, q50_c, q90_c)
        lower_c = q10_c - q50_c
        upper_c = q90_c - q50_c

        scores_t = np.maximum(lower_c - y_calib_t, y_calib_t - upper_c)
        n_t = len(scores_t)
        k_t = min(int(np.ceil((1 - alpha) * (n_t + 1))), n_t)
        Q_t = np.sort(scores_t)[k_t - 1]

        q10_corrected[mask] = q10_test[mask] - Q_t
        q90_corrected[mask] = q90_test[mask] + Q_t

    return pd.DataFrame(
        {"q10": q10_corrected, "q50": np.zeros(len(te)), "q90": q90_corrected,
         "vol_tercile": te_vol_tercile},
        index=te.index,
    )


# Decision: evaluate marginal coverage and decide
print("\n" + "=" * 80)
print("DECISION GATE")
print("=" * 80)

regime_cov = marginal_coverage["regime_stats"]["coverage"]
under_coverage = (regime_cov < 0.70).any()

if not under_coverage:
    print("✓ All regimes within 70–90% coverage")
    print("  → MARGINAL CQR STANDS (ship it)")
    USE_CONDITIONAL = False
else:
    bad_regimes = regime_cov[regime_cov < 0.70]
    print("✗ Regimes under-covering (< 70%):")
    for regime, cov in bad_regimes.items():
        print(f"    {regime}: {cov:.1%}")
    print("  → PROMOTE to VOL-TERCILE CONDITIONAL CQR")
    USE_CONDITIONAL = True

print()


DECISION GATE
✗ Regimes under-covering (< 70%):
    burst_spr21: 60.0%
    burst_spr23: 36.4%
    calm_16_21: 67.7%
  → PROMOTE to VOL-TERCILE CONDITIONAL CQR



In [24]:
if USE_CONDITIONAL:
    print("Running walk-forward evaluation with VOL-TERCILE CONDITIONAL CQR...")
    print("(This may take 10-15 minutes)")
    
    oos_results_cond = []
    for i, (tr_pos, te_pos) in enumerate(folds):
        if i % 10 == 0:
            print(f"  Fold {i}/{len(folds)}...")
        
        tr, te = model_df.iloc[tr_pos], model_df.iloc[te_pos]
        pred_df = fit_predict_with_vol_tercile_cqr(tr, te, FEATURES_VANILLA)
        pred_df["y"] = te["y"].values
        pred_df["regime"] = te["regime"].values
        
        oos_results_cond.append(pred_df)
    
    oos = pd.concat(oos_results_cond)
    cqr_coverage = compute_coverage(oos)
    
    print("\nVOL-TERCILE CONDITIONAL CQR:")
    print(f"Coverage: {cqr_coverage['overall_coverage']:.2%} (target: 80.00%)")
    print(f"Width:    {cqr_coverage['overall_width']:.4f}")
    print()
    
    print("Coverage by regime:")
    regime_stats_cond = cqr_coverage["regime_stats"].copy()
    regime_stats_cond["coverage"] = regime_stats_cond["coverage"] * 100
    print(regime_stats_cond.to_string())
    
    FINAL_COVERAGE = cqr_coverage
else:
    print("Using MARGINAL CQR (no conditioning needed)")
    FINAL_COVERAGE = marginal_coverage

print()

Running walk-forward evaluation with VOL-TERCILE CONDITIONAL CQR...
(This may take 10-15 minutes)
  Fold 0/76...
  Fold 10/76...
  Fold 20/76...
  Fold 30/76...
  Fold 40/76...
  Fold 50/76...
  Fold 60/76...
  Fold 70/76...

VOL-TERCILE CONDITIONAL CQR:
Coverage: 78.94% (target: 80.00%)
Width:    0.2420

Coverage by regime:
                coverage  count     width
regime                                   
breakout_21    65.714286     70  0.230039
burst_spr21    36.923077     65  0.234042
burst_spr23    65.454545     55  0.362132
calm_16_21     69.685039    254  0.264981
cooling_22_23  91.250000    160  0.241827
high_21_22     83.050847    295  0.326439
low_23_26      84.133916    687  0.189685



In [25]:
import joblib
from pathlib import Path

# Production quantile models: train on all-but-recent, calibrate on the trailing slice
calib_size = 252
prod_train = model_df.iloc[:-calib_size]   # adjust to your model-ready frame name
prod_calib = model_df.iloc[-calib_size:]

quantile_models = {}
for q in [0.1, 0.5, 0.9]:
    m = xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=q, **XGB_PARAMS)
    m.fit(prod_train[FEATURES_VANILLA], prod_train["y"])
    quantile_models[q] = m

# RW-centered conformal Q on the held-out calibration slice
q10c = quantile_models[0.1].predict(prod_calib[FEATURES_VANILLA])
q50c = quantile_models[0.5].predict(prod_calib[FEATURES_VANILLA])
q90c = quantile_models[0.9].predict(prod_calib[FEATURES_VANILLA])
q10c, q50c, q90c = enforce_quantile_monotonicity(q10c, q50c, q90c)

y = prod_calib["y"].values
scores = np.maximum((q10c - q50c) - y, y - (q90c - q50c))   # max(L - y, y - U)
n = len(scores) 
k = min(int(np.ceil(0.8 * (n + 1))), n)
Q = float(np.sort(scores)[k - 1])

out = Path("backend/models"); out.mkdir(parents=True, exist_ok=True)
joblib.dump(quantile_models, out / "quantile_models.joblib")
joblib.dump({"correction_q10": -Q, "correction_q90": +Q}, out / "conformal_calibration.joblib")
print(f"Saved quantile_models + conformal_calibration  (Q={Q:.4f}, band ≈ ±{Q:.3f} on top of model spread)")

Saved quantile_models + conformal_calibration  (Q=0.0100, band ≈ ±0.010 on top of model spread)


## Decision Summary

**Marginal CQR performance:**
- Overall coverage: 80.8% (target 80%)
- Best-calibrated regime: high_21_22 at 80.0% (dead on target)
- Worst regime: burst_spr23 at 36.4% (short, high-vol; little calibration history)

**Conditional CQR (vol-tercile) performance:**
- Overall coverage: 78.9%
- Improvement: none — it relocated the unevenness rather than fixing it (burst_spr23 36→65 but burst_spr21 60→37 from thin high-vol buckets) and came in below nominal. Tested, not adopted.

**Final choice:** MARGINAL

**Rationale:** Marginal CQR is closer to nominal overall (80.8% vs 78.9%) and avoids the thin-bucket instability that collapsed burst_spr21 under conditioning. Vol-tercile conditioning relocated the per-regime unevenness instead of removing it, so the simpler model with the clean marginal guarantee is the evidence-supported choice; conditioning is parked in v2-ideas.md pending more high-vol calibration history.